# GeoBrix DCS image on a dedicated DBR 19 cluster: do the Python and JVM levels work?

**What and why:** the image `mjohns797/geobrix:<version>-env6-{arm,x86}` carries the env6 wheel, its GDAL and the JAR. Its
in-image checks pass, but no cluster has run it. This probe runs the install check of `installation/dcs.mdx` ("Verify the
install") and then one heavyweight and one lightweight call per tier on a cluster started from the image.

**Expected output:** one JSON (`dbutils.notebook.exit`): the node architecture, the JAR the session sees (version, sha) and
why a level is missing (none expected), how many functions each level serves, the `gbx_rst_*` count of `SHOW FUNCTIONS`,
and per check `ok` or the error. Verdict PASS: every check `ok`, the JVM serves its default functions (L1 > 0), the JAR is
the image's one JAR (one geobrix JAR on the class path or added to the session).

**Expected time:** 2 to 5 min after the cluster start (the first start pulls about 7 GB: up to about 10 min).

**Monitoring:** one printed line per check; the job's time limit ends a stuck run; cancel from the Runs page. No retries.


In [ ]:
CORPUS = "/Volumes/geospatial_docs/geobrix/sample-data/bench-corpus/rows"
N_TILES = 64


In [ ]:
import json
import os
import platform
from collections import Counter

from databricks.labs import gbx
from databricks.labs.gbx.manager import capabilities

RESULT = {"arch": platform.machine(), "checks": {}}
caps = capabilities.detect(spark, True)
RESULT["jar"] = None if caps.jar is None else {"version": caps.jar.version, "sha256": caps.jar.sha256}
RESULT["jar_reason"], RESULT["gdal_reason"], RESULT["jvm_gdal"] = caps.jar_reason, caps.gdal_reason, caps.jvm_gdal
# every geobrix JAR the driver JVM sees: on its class path (the image's /databricks/jars) and added to the session
cp = spark._jvm.java.lang.System.getProperty("java.class.path").split(":")
added = spark.sparkContext._jsc.sc().listJars().mkString(",").split(",")
jars = sorted({os.path.basename(j) for j in cp + added if "geobrix" in j.lower() and j.endswith(".jar")})
RESULT["geobrix_jars"] = jars

# the install check of installation/dcs.mdx ("Verify the install")
choices = gbx.register(spark)
RESULT["levels"] = dict(Counter(c.level for c in choices.values()))
# the doc's SHOW FUNCTIONS line, kept from stopping the run: a parse error is recorded with the other spellings
try:
    RESULT["show_functions_rst"] = spark.sql("SHOW FUNCTIONS LIKE 'gbx_rst_*'").count()
except Exception as e:  # noqa: BLE001
    RESULT["show_functions_rst"] = None
    RESULT["checks"]["show_functions_like"] = {"ok": False, "error": f"{type(e).__name__}: {e}"[:600]}
    for label, q in {"like_star": "SHOW FUNCTIONS LIKE 'gbx_rst*'", "like_builtin": "SHOW FUNCTIONS LIKE 'abs*'", "list_functions": None}.items():
        try:
            n = len([f for f in spark.catalog.listFunctions() if f.name.startswith("gbx_rst_")]) if q is None else spark.sql(q).count()
            RESULT["checks"]["show_functions_" + label] = {"ok": True, "n": n}
        except Exception as e2:  # noqa: BLE001
            RESULT["checks"]["show_functions_" + label] = {"ok": False, "error": f"{type(e2).__name__}: {e2}"[:400]}
    print("show_functions", json.dumps({k: v for k, v in RESULT["checks"].items() if k.startswith("show_functions")}), flush=True)
print(json.dumps(RESULT), flush=True)


In [ ]:
def check(name, sql, want=None):
    try:
        row = spark.sql(sql).collect()[0].asDict()
        ok = want is None or all(row.get(k) == v for k, v in want.items())
        RESULT["checks"][name] = {"ok": ok, "row": {k: str(v)[:200] for k, v in row.items()}}
    except Exception as e:  # noqa: BLE001
        RESULT["checks"][name] = {"ok": False, "error": f"{type(e).__name__}: {e}"[:600]}
    print(name, json.dumps(RESULT["checks"][name]), flush=True)


paths = [f.path for f in dbutils.fs.ls(CORPUS) if f.path.endswith(".tif")][:N_TILES]
spark.read.format("binaryFile").load(paths).createOrReplaceTempView("probe_bytes")
spark.read.format("raster_gbx").load(CORPUS, filterRegex="|".join(p.rsplit("/", 1)[-1] for p in paths)).createOrReplaceTempView("probe_tiles")
heavy = {n: c.level for n, c in choices.items() if n in ("gbx_rst_fromcontent", "gbx_rst_width", "gbx_st_transformcrs")}
RESULT["heavy_levels"] = heavy
print("heavy levels", heavy, flush=True)
# JVM level: bytes to a tile and its width; a vector transform
check("jvm_fromcontent_width", f"SELECT count(*) AS n, count(gbx_rst_width(gbx_rst_fromcontent(content, 'GTiff'))) AS n_out FROM probe_bytes",
      {"n": len(paths), "n_out": len(paths)})
check("jvm_st_transformcrs", "SELECT st_astext(st_geomfromwkb(gbx_st_transformcrs('POINT(530000 180000)', 'EPSG:4326', 'EPSG:27700'))) AS g")
# light level: the GDAL engine's slope and an Arrow accessor on the reader's tiles (virtual tiles go to light)
check("light_slope", "SELECT count(*) AS n, count(gbx_rst_slope(tile).raster) AS n_out FROM probe_tiles", {"n": len(paths), "n_out": len(paths)})
check("light_avg", "SELECT count(gbx_rst_avg(tile)) AS n_out FROM probe_tiles", {"n_out": len(paths)})
check("reader_raster_gbx", "SELECT count(*) AS n FROM probe_tiles", {"n": len(paths)})
RESULT["verdict"] = (
    "PASS"
    if all(c["ok"] for c in RESULT["checks"].values()) and RESULT["levels"].get("L1", 0) > 0 and len(jars) == 1 and RESULT["jar"] is not None
    else "FAIL"
)
print("verdict", RESULT["verdict"], flush=True)


In [ ]:
dbutils.notebook.exit(json.dumps(RESULT))
